# DICTA Ablation Study: MobileNetV3-Small vs EfficientNet-B0

This notebook is a focused ablation study comparing only two backbones for the shared multi-task pipeline:
- **MobileNetV3-Small** (edge-first candidate)
- **EfficientNet-B0** (accuracy-first candidate)

## Objective
Identify which backbone is better for DICTA UAV deployment when balancing:
1. Segmentation quality (**primary metric: mIoU**)
2. Edge feasibility (P95 latency, FPS, model size)
3. Practical deployment constraints

## DICTA Constraints
- mIoU >= 0.80
- P95 latency < 100 ms
- model size < 2 MB

## Expected Outcome
A single clear recommendation backed by quantitative evidence and trade-off analysis.

In [5]:
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List
import json

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torchvision.models as tv_models
from scipy import stats

import sys
sys.path.append('../../')

from models.unet_segmentation import UltraOptimizedFireNet

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path('../../')
OUTPUT_DIR = ROOT / 'data' / 'processed' / 'Output' / 'DICTA'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Ablation scope: MobileNetV3-Small vs EfficientNet-B0')
print(f'Device: {DEVICE}')
print(f'Output dir: {OUTPUT_DIR.resolve()}')

Ablation scope: MobileNetV3-Small vs EfficientNet-B0
Device: cpu
Output dir: C:\SPJAIN\BushFire-Detection\data\processed\Output\DICTA


## 1) Setup and Experiment Protocol

This section defines reproducible settings and a unified result schema.

Protocol:
- Same input size (224x224, 4-channel RGBT)
- Same metric definitions across both models
- Separate reporting of:
  - **System benchmark** (random input, architecture-level speed/size)
  - **Validation benchmark** (mIoU, Fire-IoU, F1 from real runs)

Final ranking is based on DICTA pass/fail + highest mIoU among passing models.

In [6]:
class EfficientNetB0DualHead(nn.Module):
    def __init__(self, num_classes: int = 2, num_seg_classes: int = 2):
        super().__init__()
        backbone = tv_models.efficientnet_b0(weights=None)

        # Adapt first conv: 3 -> 4 channels (RGBT).
        old_conv = backbone.features[0][0]
        new_conv = nn.Conv2d(
            4,
            old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False,
        )
        nn.init.kaiming_normal_(new_conv.weight, mode='fan_out', nonlinearity='relu')
        backbone.features[0][0] = new_conv

        self.backbone = backbone.features
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        with torch.no_grad():
            feat = self.backbone(torch.randn(1, 4, 224, 224))
            c = feat.shape[1]

        self.classifier = nn.Sequential(
            nn.Linear(c, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )

        self.seg_head = nn.Sequential(
            nn.Conv2d(c, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, kernel_size=8, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, num_seg_classes, kernel_size=1),
        )

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        f = self.backbone(x)
        cls = self.avgpool(f).flatten(1)
        cls = self.classifier(cls)

        seg = self.seg_head(f)
        seg = torch.nn.functional.interpolate(seg, size=(224, 224), mode='bilinear', align_corners=False)

        return {'classification': cls, 'segmentation': seg}


@dataclass
class ModelBenchmark:
    model_name: str
    params_million: float
    model_size_mb: float
    latency_p95_ms: float
    fps: float


def get_model_size_mb(model: nn.Module) -> float:
    total_params = sum(p.numel() for p in model.parameters())
    return (total_params * 4) / (1024 ** 2)


def benchmark_latency(
    model: nn.Module,
    device: str = 'cpu',
    runs: int = 80,
    warmup: int = 20,
    input_shape=(1, 4, 224, 224),
    return_samples: bool = False,
) -> Dict[str, float]:
    model = model.to(device).eval()
    x = torch.randn(*input_shape, device=device)

    with torch.no_grad():
        for _ in range(warmup):
            _ = model(x)

    if device == 'cuda':
        torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for _ in range(runs):
            if device == 'cuda':
                start = torch.cuda.Event(enable_timing=True)
                end = torch.cuda.Event(enable_timing=True)
                start.record()
                _ = model(x)
                end.record()
                torch.cuda.synchronize()
                times.append(start.elapsed_time(end))  # ms
            else:
                import time
                t0 = time.perf_counter()
                _ = model(x)
                t1 = time.perf_counter()
                times.append((t1 - t0) * 1000.0)

    arr = np.array(times, dtype=np.float64)
    p95 = float(np.percentile(arr, 95))
    fps = float(1000.0 / p95) if p95 > 0 else 0.0

    result = {'latency_p95_ms': p95, 'fps': fps}
    if return_samples:
        result['samples_ms'] = arr
    return result

## 2) Validation Metrics (from training/evaluation runs)

Update the values below from your actual run logs/checkpoints if needed.

Required fields:
- `miou` (primary)
- `fire_iou`, `dice`, `f1_cls`
- `latency_p95_ms` (real deployment benchmark, if available)

If `latency_p95_ms` is not available from validation logs, the notebook will fall back to architecture benchmark latency.

In [12]:
# 1) Build both models
model_mobilenet = UltraOptimizedFireNet()
model_efficientnet = EfficientNetB0DualHead()

# 2) Benchmark architecture-level efficiency
benchmarks: List[ModelBenchmark] = []
latency_samples = {}
for name, model in [
    ('MobileNetV3-Small (UltraOptimizedFireNet)', model_mobilenet),
    ('EfficientNet-B0 (Dual-Head)', model_efficientnet),
]:
    size_mb = get_model_size_mb(model)
    params_m = sum(p.numel() for p in model.parameters()) / 1e6
    perf = benchmark_latency(model, device=DEVICE, return_samples=True)
    latency_samples[name] = perf['samples_ms']
    benchmarks.append(
        ModelBenchmark(
            model_name=name,
            params_million=params_m,
            model_size_mb=size_mb,
            latency_p95_ms=perf['latency_p95_ms'],
            fps=perf['fps'],
        )
    )

df_system = pd.DataFrame([b.__dict__ for b in benchmarks])
df_system

,model_name,params_million,model_size_mb,latency_p95_ms,fps
0,MobileNetV3-Small (UltraOptimizedFireNet),0.458300,1.748276,9.375625,106.659556
1,EfficientNet-B0 (Dual-Head),6.171232,23.541382,28.403450,35.206991


In [13]:
# Load validation metrics from tracked artifacts (no hardcoded placeholders)
ablation_csv_path = OUTPUT_DIR / 'ablation_backbone_mobilenet_vs_efficientnet.csv'
dicta_metrics_path = ROOT / 'data' / 'processed' / 'Output' / 'Classification' / 'shared_backbone_metrics.json'

if not ablation_csv_path.exists():
    raise FileNotFoundError(f'Missing ablation input file: {ablation_csv_path}')

df_raw = pd.read_csv(ablation_csv_path)

# Normalize potential legacy schemas (_val/_sys columns from previous exports)
def pick_col(frame, candidates, default=np.nan):
    for c in candidates:
        if c in frame.columns:
            return frame[c]
    return pd.Series([default] * len(frame))

df_val = pd.DataFrame({
    'model_name': pick_col(df_raw, ['model_name']),
    'miou': pick_col(df_raw, ['miou']),
    'fire_iou': pick_col(df_raw, ['fire_iou']),
    'dice': pick_col(df_raw, ['dice']),
    'f1_cls': pick_col(df_raw, ['f1_cls']),
    'latency_p95_ms': pick_col(df_raw, ['latency_p95_ms', 'latency_p95_ms_val', 'latency_p95_ms_sys']),
    'model_size_mb': pick_col(df_raw, ['model_size_mb', 'model_size_mb_val', 'model_size_mb_sys']),
    'params_million': pick_col(df_raw, ['params_million', 'params_million_val', 'params_million_sys']),
    'fps': pick_col(df_raw, ['fps', 'fps_val', 'fps_sys']),
})

required_cols = {'model_name', 'miou', 'fire_iou', 'dice', 'f1_cls', 'latency_p95_ms'}
if not required_cols.issubset(df_val.columns):
    raise ValueError(f'Missing required columns in normalized dataframe: {required_cols - set(df_val.columns)}')

# Refresh DICTA/MobileNet row with latest shared-backbone artifact if available
if dicta_metrics_path.exists():
    with open(dicta_metrics_path, 'r') as f:
        dicta_metrics = json.load(f)
    mobile_mask = df_val['model_name'].str.contains('MobileNetV3-Small', case=False, na=False)
    if mobile_mask.any():
        df_val.loc[mobile_mask, 'miou'] = float(dicta_metrics['segmentation_metrics']['miou'])
        df_val.loc[mobile_mask, 'f1_cls'] = float(dicta_metrics['classification_metrics']['f1'])
        df_val.loc[mobile_mask, 'latency_p95_ms'] = float(dicta_metrics['latency_ms']['p95'])
        df_val.loc[mobile_mask, 'model_size_mb'] = float(dicta_metrics['model']['size_mb'])
        df_val.loc[mobile_mask, 'params_million'] = float(dicta_metrics['model']['parameters']) / 1e6
        df_val.loc[mobile_mask, 'fps'] = float(dicta_metrics['latency_ms']['fps'])

# Merge system benchmark metrics and only fill missing canonical values
df_final = df_val.merge(
    df_system[['model_name', 'params_million', 'model_size_mb', 'latency_p95_ms', 'fps']],
    on='model_name',
    how='left',
    suffixes=('', '_sys')
)

for col in ['params_million', 'model_size_mb', 'latency_p95_ms', 'fps']:
    sys_col = f'{col}_sys'
    if sys_col in df_final.columns:
        df_final[col] = df_final[col].fillna(df_final[sys_col])
        df_final.drop(columns=[sys_col], inplace=True)

# If FPS missing but latency exists, compute it consistently
missing_fps = df_final['fps'].isna() & df_final['latency_p95_ms'].notna() & (df_final['latency_p95_ms'] > 0)
df_final.loc[missing_fps, 'fps'] = 1000.0 / df_final.loc[missing_fps, 'latency_p95_ms']

# DICTA pass/fail criteria
df_final['dicta_pass'] = (
    (df_final['miou'] >= 0.80)
    & (df_final['latency_p95_ms'] < 100.0)
    & (df_final['model_size_mb'] < 2.0)
)

# Winner logic: among passing models, pick highest mIoU then lower latency
passing = df_final[df_final['dicta_pass']].copy()
if len(passing) > 0:
    winner = passing.sort_values(['miou', 'latency_p95_ms'], ascending=[False, True]).iloc[0]
else:
    winner = df_final.sort_values(['miou', 'latency_p95_ms'], ascending=[False, True]).iloc[0]

# Compute interpretable trade-offs
df_ranked = df_final.sort_values(['dicta_pass', 'miou', 'latency_p95_ms'], ascending=[False, False, True]).reset_index(drop=True)
display(df_ranked[['model_name', 'miou', 'fire_iou', 'dice', 'f1_cls', 'model_size_mb', 'latency_p95_ms', 'fps', 'dicta_pass']])

print('\nRecommended backbone for DICTA:')
print(winner[['model_name', 'miou', 'latency_p95_ms', 'model_size_mb', 'dicta_pass']])

# Claim validation for ablation narrative
other = df_ranked[df_ranked['model_name'] != winner['model_name']].iloc[0]
ablation_claims = pd.DataFrame([
    {
        'claim': 'Selected winner passes DICTA constraints',
        'status': 'PASS' if bool(winner['dicta_pass']) else 'FAIL',
        'actual': bool(winner['dicta_pass']),
        'reference': True
    },
    {
        'claim': 'Winner has highest mIoU among passing models',
        'status': 'PASS' if (len(passing) == 0 or winner['miou'] >= passing['miou'].max()) else 'FAIL',
        'actual': float(winner['miou']),
        'reference': float(passing['miou'].max()) if len(passing) > 0 else float(df_final['miou'].max())
    },
    {
        'claim': 'Winner latency is deployment-feasible (<100 ms)',
        'status': 'PASS' if float(winner['latency_p95_ms']) < 100.0 else 'FAIL',
        'actual': float(winner['latency_p95_ms']),
        'reference': 100.0
    },
    {
        'claim': 'Winner model size is edge-feasible (<2 MB)',
        'status': 'PASS' if float(winner['model_size_mb']) < 2.0 else 'FAIL',
        'actual': float(winner['model_size_mb']),
        'reference': 2.0
    },
])
display(ablation_claims)

# Save outputs
out_csv = OUTPUT_DIR / 'ablation_backbone_mobilenet_vs_efficientnet.csv'
df_final.to_csv(out_csv, index=False)

summary_json = OUTPUT_DIR / 'ablation_study_summary.json'
summary_payload = {
    'winner': winner.to_dict(),
    'runner_up': other.to_dict(),
    'claims': ablation_claims.to_dict(orient='records'),
    'submission_ready': bool((ablation_claims['status'] == 'PASS').all()),
}
with open(summary_json, 'w') as f:
    json.dump(summary_payload, f, indent=2)

print(f'\nSaved comparison table: {out_csv}')
print(f'Saved ablation summary: {summary_json}')

,model_name,miou,fire_iou,dice,f1_cls,model_size_mb,latency_p95_ms,fps,dicta_pass
0,MobileNetV3-Small (UltraOptimizedFireNet),0.8296,0.8012,0.8891,0.7734,1.748300,10.05,94.300000,True
1,EfficientNet-B0 (Dual-Head),0.8341,0.8048,0.8920,0.7849,23.541382,31.40,31.847134,False



Recommended backbone for DICTA:
model_name        MobileNetV3-Small (UltraOptimizedFireNet)
miou                                                 0.8296
latency_p95_ms                                        10.05
model_size_mb                                        1.7483
dicta_pass                                             True
Name: 0, dtype: object


,claim,status,actual,reference
0,Selected winner passes DICTA constraints,PASS,True,True
1,Winner has highest mIoU among passing models,PASS,0.8296,0.8296
2,Winner latency is deployment-feasible (<100 ms),PASS,10.05,100.0
3,Winner model size is edge-feasible (<2 MB),PASS,1.7483,2.0



Saved comparison table: ..\..\data\processed\Output\DICTA\ablation_backbone_mobilenet_vs_efficientnet.csv
Saved ablation summary: ..\..\data\processed\Output\DICTA\ablation_study_summary.json


## 3) Interpretation Guide

Use this checklist when writing the DICTA report section:

1. **mIoU-first decision**
- Compare mIoU difference between the two backbones.
- Explain boundary quality impact (small, irregular fire regions).

2. **Edge feasibility decision**
- Compare P95 latency, FPS, and model size.
- Explicitly state DICTA pass/fail per model.

3. **Final recommendation**
- If both pass DICTA constraints: choose higher mIoU.
- If one fails constraints: select the passing model and justify trade-off.

4. **Deployment rationale**
- Link final choice to UAV real-time requirement and robustness.

## 4) Final Conclusion (Auto-generated Summary Target)

After running all cells, use the generated winner row to finalize this statement:

- Which backbone wins under DICTA constraints
- How much mIoU gain/loss it brings
- Whether latency and model size remain edge-feasible

**Expected practical outcome**: MobileNetV3-Small is typically the safer edge deployment choice unless EfficientNet-B0 provides significantly better mIoU while still meeting deployment constraints.

## 5) Component Justification (from Stage-2 Classification Results)

This section validates which modality/fusion component is justified before backbone selection.
It uses tracked results from `data/processed/Output/Classification/stage2_results_table.csv`.

In [14]:
stage2_csv = ROOT / 'data' / 'processed' / 'Output' / 'Classification' / 'stage2_results_table.csv'
if not stage2_csv.exists():
    raise FileNotFoundError(f'Missing component analysis input: {stage2_csv}')

df_components = pd.read_csv(stage2_csv)
df_components = df_components.sort_values('F1', ascending=False).reset_index(drop=True)
display(df_components[['Model', 'Acc', 'Precision', 'Recall', 'F1', 'AUC', 'FP', 'FN']])

best_component = df_components.iloc[0]
worst_component = df_components.iloc[-1]
print('Component-level justification:')
print(f"  Best component strategy: {best_component['Model']} (F1={best_component['F1']:.4f})")
print(f"  Worst component strategy: {worst_component['Model']} (F1={worst_component['F1']:.4f})")
print(f"  F1 gap: {(best_component['F1'] - worst_component['F1']):.4f}")

,Model,Acc,Precision,Recall,F1,AUC,FP,FN
0,Thermal-only,0.997714,0.999399,0.997002,0.998199,0.999991,3,15
1,EarlyFusion_4ch,0.995301,0.998594,0.994004,0.996294,0.999952,7,30
2,RGB-only (Stage 1),0.973584,0.975035,0.983610,0.979303,0.994416,126,82
3,LateFusion_Standard,0.375159,0.517199,0.249450,0.336570,0.420977,1165,3755
4,LateFusion_CostSens,0.375159,0.517199,0.249450,0.336570,0.420977,1165,3755


Component-level justification:
  Best component strategy: Thermal-only (F1=0.9982)
  Worst component strategy: LateFusion_CostSens (F1=0.3366)
  F1 gap: 0.6616


## 6) Loss-Weight Sensitivity + Statistical Significance

This section completes the ablation by:
- Running a multi-task loss-weight sweep (`w_seg` vs `w_cls`) using real metrics and deployment penalties.
- Testing latency difference significance between MobileNetV3-Small and EfficientNet-B0 using Mann-Whitney U test.

In [15]:
# Build a compact objective for loss-weight analysis
weights = np.round(np.linspace(0.1, 0.9, 9), 2)
rows = []
for w_seg in weights:
    w_cls = 1.0 - w_seg
    for _, r in df_final.iterrows():
        seg_loss = 1.0 - float(r['miou'])
        cls_loss = 1.0 - float(r['f1_cls'])
        # Penalize non-deployable models to reflect DICTA constraints in objective search
        deploy_penalty = 0.5 if not bool(r['dicta_pass']) else 0.0
        weighted_score = (w_seg * seg_loss) + (w_cls * cls_loss) + deploy_penalty
        rows.append({
            'w_seg': w_seg,
            'w_cls': w_cls,
            'model_name': r['model_name'],
            'seg_loss': seg_loss,
            'cls_loss': cls_loss,
            'deploy_penalty': deploy_penalty,
            'weighted_score': weighted_score,
        })

df_weight = pd.DataFrame(rows)
best_by_weight = df_weight.loc[df_weight.groupby(['w_seg'])['weighted_score'].idxmin()].sort_values('w_seg')
display(best_by_weight[['w_seg', 'w_cls', 'model_name', 'weighted_score', 'deploy_penalty']])

# Recommendation stability across weight sweep
winner_counts = best_by_weight['model_name'].value_counts()
print('Loss-weight sweep winner counts:')
print(winner_counts.to_string())

# Statistical significance on latency samples (captured in cell 6)
m_key = 'MobileNetV3-Small (UltraOptimizedFireNet)'
e_key = 'EfficientNet-B0 (Dual-Head)'
if m_key not in latency_samples or e_key not in latency_samples:
    raise RuntimeError('Latency samples not found. Re-run the system benchmark cell first.')

m_latency = np.array(latency_samples[m_key], dtype=np.float64)
e_latency = np.array(latency_samples[e_key], dtype=np.float64)
u_stat, p_value = stats.mannwhitneyu(m_latency, e_latency, alternative='two-sided')

# Effect size (Cliff's delta)
def cliffs_delta(a, b):
    gt = np.sum(a[:, None] > b[None, :])
    lt = np.sum(a[:, None] < b[None, :])
    return (gt - lt) / (len(a) * len(b))

delta = float(cliffs_delta(m_latency, e_latency))
median_diff = float(np.median(e_latency) - np.median(m_latency))

sig_df = pd.DataFrame([
    {
        'test': 'Mann-Whitney U (latency)',
        'u_stat': float(u_stat),
        'p_value': float(p_value),
        'significant_0p05': bool(p_value < 0.05),
        'cliffs_delta_mobile_vs_efficient': delta,
        'median_latency_gap_ms_eff_minus_mobile': median_diff,
    }
])
display(sig_df)

print('Interpretation:')
print(f"  MobileNet median latency advantage (ms): {median_diff:.3f}")
print(f"  p-value: {p_value:.6f} -> {'significant' if p_value < 0.05 else 'not significant'}")

sig_df.to_csv(OUTPUT_DIR / 'ablation_latency_significance.csv', index=False)
best_by_weight.to_csv(OUTPUT_DIR / 'ablation_loss_weight_sensitivity.csv', index=False)
print(f"Saved: {OUTPUT_DIR / 'ablation_latency_significance.csv'}")
print(f"Saved: {OUTPUT_DIR / 'ablation_loss_weight_sensitivity.csv'}")

,w_seg,w_cls,model_name,weighted_score,deploy_penalty
0,0.1,0.9,MobileNetV3-Small (UltraOptimizedFireNet),0.22098,0.0
2,0.2,0.8,MobileNetV3-Small (UltraOptimizedFireNet),0.21536,0.0
4,0.3,0.7,MobileNetV3-Small (UltraOptimizedFireNet),0.20974,0.0
6,0.4,0.6,MobileNetV3-Small (UltraOptimizedFireNet),0.20412,0.0
8,0.5,0.5,MobileNetV3-Small (UltraOptimizedFireNet),0.19850,0.0
10,0.6,0.4,MobileNetV3-Small (UltraOptimizedFireNet),0.19288,0.0
12,0.7,0.3,MobileNetV3-Small (UltraOptimizedFireNet),0.18726,0.0
14,0.8,0.2,MobileNetV3-Small (UltraOptimizedFireNet),0.18164,0.0
16,0.9,0.1,MobileNetV3-Small (UltraOptimizedFireNet),0.17602,0.0


Loss-weight sweep winner counts:
model_name
MobileNetV3-Small (UltraOptimizedFireNet)    9


,test,u_stat,p_value,significant_0p05,cliffs_delta_mobile_vs_efficient,median_latency_gap_ms_eff_minus_mobile
0,Mann-Whitney U (latency),0.0,9.385827e-28,True,-1.0,16.2907


Interpretation:
  MobileNet median latency advantage (ms): 16.291
  p-value: 0.000000 -> significant
Saved: ..\..\data\processed\Output\DICTA\ablation_latency_significance.csv
Saved: ..\..\data\processed\Output\DICTA\ablation_loss_weight_sensitivity.csv


## 7) Final Ablation Verdict (Auto-generated)

This section consolidates all critical ablation evidence into one decision package:
- Real validation/system metrics (MobileNetV3-Small vs EfficientNet-B0)
- Loss-weight sweep stability
- Component-level justification
- Statistical significance evidence (latency)

The output is exported as a machine-readable evidence bundle for reporting/review.

In [16]:
# Consolidate all ablation evidence into a final decision bundle
actual_results = df_final[['model_name', 'miou', 'fire_iou', 'dice', 'f1_cls', 'model_size_mb', 'latency_p95_ms', 'fps', 'dicta_pass']].copy()
actual_results = actual_results.sort_values(['dicta_pass', 'miou', 'latency_p95_ms'], ascending=[False, False, True]).reset_index(drop=True)

overall_winner = actual_results.iloc[0]
runner_up = actual_results.iloc[1] if len(actual_results) > 1 else actual_results.iloc[0]

loss_weight_winner = winner_counts.index[0] if len(winner_counts) > 0 else str(overall_winner['model_name'])
loss_weight_stability = float(winner_counts.iloc[0] / winner_counts.sum()) if len(winner_counts) > 0 else 1.0

lat_sig = sig_df.iloc[0]

verdict_rows = [
    {
        'section': 'backbone_recommendation',
        'winner': str(overall_winner['model_name']),
        'runner_up': str(runner_up['model_name']),
        'winner_miou': float(overall_winner['miou']),
        'runner_up_miou': float(runner_up['miou']),
        'miou_gap_winner_minus_runnerup': float(overall_winner['miou'] - runner_up['miou']),
        'winner_latency_p95_ms': float(overall_winner['latency_p95_ms']),
        'winner_size_mb': float(overall_winner['model_size_mb']),
        'winner_dicta_pass': bool(overall_winner['dicta_pass']),
    },
    {
        'section': 'loss_weight_sensitivity',
        'winner': str(loss_weight_winner),
        'weight_sweep_stability_ratio': loss_weight_stability,
        'num_weight_points': int(winner_counts.sum()) if len(winner_counts) > 0 else 1,
        'details': winner_counts.to_dict(),
    },
    {
        'section': 'component_justification',
        'best_component': str(best_component['Model']),
        'worst_component': str(worst_component['Model']),
        'best_component_f1': float(best_component['F1']),
        'worst_component_f1': float(worst_component['F1']),
        'component_f1_gap': float(best_component['F1'] - worst_component['F1']),
    },
    {
        'section': 'latency_significance',
        'test': str(lat_sig['test']),
        'p_value': float(lat_sig['p_value']),
        'significant_0p05': bool(lat_sig['significant_0p05']),
        'cliffs_delta_mobile_vs_efficient': float(lat_sig['cliffs_delta_mobile_vs_efficient']),
        'median_latency_gap_ms_eff_minus_mobile': float(lat_sig['median_latency_gap_ms_eff_minus_mobile']),
    },
]

verdict_df = pd.DataFrame(verdict_rows)
display(actual_results)
display(verdict_df)

submission_ready = bool((ablation_claims['status'] == 'PASS').all())
print('Final ablation verdict summary:')
print(f"  Recommended backbone: {overall_winner['model_name']}")
print(f"  DICTA pass: {bool(overall_winner['dicta_pass'])}")
print(f"  mIoU (winner): {float(overall_winner['miou']):.4f}")
print(f"  P95 latency (winner): {float(overall_winner['latency_p95_ms']):.3f} ms")
print(f"  Model size (winner): {float(overall_winner['model_size_mb']):.4f} MB")
print(f"  Loss-weight stability: {loss_weight_stability:.2%}")
print(f"  Latency significance p-value: {float(lat_sig['p_value']):.6f}")
print(f"  Submission ready: {submission_ready}")

evidence_bundle = {
    'actual_results': actual_results.to_dict(orient='records'),
    'ablation_claims': ablation_claims.to_dict(orient='records'),
    'loss_weight_winner_counts': winner_counts.to_dict(),
    'latency_significance': sig_df.to_dict(orient='records'),
    'verdict': verdict_rows,
    'submission_ready': submission_ready,
}

bundle_path = OUTPUT_DIR / 'ablation_evidence_bundle.json'
verdict_csv = OUTPUT_DIR / 'ablation_final_verdict.csv'

with open(bundle_path, 'w') as f:
    json.dump(evidence_bundle, f, indent=2)
verdict_df.to_csv(verdict_csv, index=False)

print(f"Saved: {bundle_path}")
print(f"Saved: {verdict_csv}")

,model_name,miou,fire_iou,dice,f1_cls,model_size_mb,latency_p95_ms,fps,dicta_pass
0,MobileNetV3-Small (UltraOptimizedFireNet),0.8296,0.8012,0.8891,0.7734,1.748300,10.05,94.300000,True
1,EfficientNet-B0 (Dual-Head),0.8341,0.8048,0.8920,0.7849,23.541382,31.40,31.847134,False


,section,winner,runner_up,winner_miou,runner_up_miou,miou_gap_winner_minus_runnerup,winner_latency_p95_ms,winner_size_mb,winner_dicta_pass,weight_sweep_stability_ratio,...,best_component,worst_component,best_component_f1,worst_component_f1,component_f1_gap,test,p_value,significant_0p05,cliffs_delta_mobile_vs_efficient,median_latency_gap_ms_eff_minus_mobile
0,backbone_recommendation,MobileNetV3-Small (UltraOptimizedFireNet),EfficientNet-B0 (Dual-Head),0.8296,0.8341,-0.0045,10.05,1.7483,True,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,loss_weight_sensitivity,MobileNetV3-Small (UltraOptimizedFireNet),NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,component_justification,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Thermal-only,LateFusion_CostSens,0.998199,0.33657,0.661629,NaN,NaN,NaN,NaN,NaN
3,latency_significance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Mann-Whitney U (latency),9.385827e-28,True,-1.0,16.2907


Final ablation verdict summary:
  Recommended backbone: MobileNetV3-Small (UltraOptimizedFireNet)
  DICTA pass: True
  mIoU (winner): 0.8296
  P95 latency (winner): 10.050 ms
  Model size (winner): 1.7483 MB
  Loss-weight stability: 100.00%
  Latency significance p-value: 0.000000
  Submission ready: True
Saved: ..\..\data\processed\Output\DICTA\ablation_evidence_bundle.json
Saved: ..\..\data\processed\Output\DICTA\ablation_final_verdict.csv
